# FREUID Challenge — Diagnostic: Format-Correlated Pipeline Signatures (Option A)

**Context:** Grad-CAM on the v4/fold0 model showed a document-*format* split, not a face-vs-not
split: all 8 most-confident-**genuine** examples were `MAURITIUS/ID` (the only ID-card-format
type), all 8 most-confident-**fraud** and all 8 **borderline** examples were driving-license-format
types (Guinea/Benin/Mozambique/Egypt DL). The fraud-rate crosstab ruled out class-imbalance as the
cause — every type sits at ~40-50% fraud. So if there's a shortcut here, it isn't "this type is
usually fraud/genuine," it's something lower-level that differs **between the ID-card rendering
pipeline and the driving-license rendering pipeline**, independent of the true label.

**This notebook checks that directly.** No model, no GPU, no checkpoints — just pixel-level and
file-level statistics computed on a sample of **confirmed-genuine** (`label=0`) training images,
compared across document type. If `MAURITIUS/ID` genuine images look statistically different from
DL-format genuine images on properties like resolution, JPEG compression signature, noise level, or
high-frequency content — properties that have *nothing to do with fraud* — that's strong evidence
the model may be keying off pipeline fingerprints rather than genuine forgery cues, and it tells us
what kind of augmentation would actually target the problem.

A secondary pass repeats the same comparison on `label=1` (fraud) images, to check whether any
found signature is present regardless of label (further supporting a pipeline-artifact story) or is
specific to the genuine class only.

**Runtime: ~10-15 min, CPU only.** No `torch`, no checkpoints, no accelerator settings to worry
about — this is deliberately independent of everything that broke last time.


## 1. Environment Setup

In [ ]:
import os
import time
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import scipy.stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Environment ready. No torch/GPU dependency in this notebook.')
print(f'OpenCV : {cv2.__version__}')
print(f'Pillow : {Image.__version__ if hasattr(Image, "__version__") else "n/a"}')

## 2. Config

In [ ]:
DATA_DIR          = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
TRAIN_LABELS_FILE = 'train_labels.csv'
OUTPUT_DIR        = '/kaggle/working/pipeline_diagnostic'

N_PER_TYPE_GENUINE = 300   # sampled per document type, label=0 pass
N_PER_TYPE_FRAUD    = 150   # smaller secondary pass, label=1

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Config loaded. Sampling {N_PER_TYPE_GENUINE}/type (genuine) + '
      f'{N_PER_TYPE_FRAUD}/type (fraud, secondary pass).')

## 3. Data Loading and Stratified Sampling

In [ ]:
data_dir = Path(DATA_DIR)

labels_path = data_dir / TRAIN_LABELS_FILE
if not labels_path.exists():
    raise FileNotFoundError(f'Labels file not found: {labels_path}')

train_df = pd.read_csv(labels_path)
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

print(f'Train shape: {train_df.shape}')
print(train_df['type'].value_counts())


def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    c = base_dir / rel_path
    if c.exists(): return c
    parts = Path(rel_path).parts
    if len(parts) > 1:
        d = base_dir / parts[0] / rel_path
        if d.exists(): return d
    f = base_dir / Path(rel_path).name
    if f.exists(): return f
    return c


def stratified_sample(df: pd.DataFrame, label_value: int, n_per_type: int) -> pd.DataFrame:
    subset = df[df['label'] == label_value]
    parts = []
    for doc_type, grp in subset.groupby('type'):
        n = min(n_per_type, len(grp))
        parts.append(grp.sample(n=n, random_state=SEED))
    return pd.concat(parts).reset_index(drop=True)


sample_genuine = stratified_sample(train_df, label_value=0, n_per_type=N_PER_TYPE_GENUINE)
sample_fraud   = stratified_sample(train_df, label_value=1, n_per_type=N_PER_TYPE_FRAUD)

print(f'\nGenuine sample: {len(sample_genuine)} rows')
print(sample_genuine['type'].value_counts())
print(f'\nFraud sample: {len(sample_fraud)} rows')
print(sample_fraud['type'].value_counts())

## 4. Low-Level Image Statistics Functions

All of these are properties that have **nothing to do with fraud** — resolution, compression signature, noise level, high-frequency content. If they differ systematically by document *format* even among confirmed-genuine images, that's a rendering/capture pipeline signature the model could latch onto instead of real forgery cues.

In [ ]:
def estimate_jpeg_quant_mean(pil_img: Image.Image) -> float:
    """Mean value of the luminance JPEG quantization table. Higher = more lossy compression
    (lower effective quality). Not an exact JPEG-quality-percentage inversion, but a robust,
    directly comparable proxy across images -- and that's all we need here."""
    try:
        if pil_img.format != 'JPEG':
            return np.nan
        qt = pil_img.quantization
        if not qt or 0 not in qt:
            return np.nan
        return float(np.mean(qt[0]))
    except Exception:
        return np.nan


def compute_noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer's fast noise estimator: convolve with a Laplacian-like kernel that has zero
    response on smooth/linear regions, so the residual energy approximates additive noise."""
    H, W = gray.shape
    if H < 3 or W < 3:
        return np.nan
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float64)
    conv = cv2.filter2D(gray.astype(np.float64), -1, kernel)
    sigma = np.sqrt(np.pi / 2) * np.sum(np.abs(conv)) / (6 * (W - 2) * (H - 2))
    return float(sigma)


def compute_fft_high_freq_ratio(gray: np.ndarray, low_frac: float = 0.1) -> float:
    """Fraction of FFT magnitude energy OUTSIDE a low-frequency central box. Higher = more
    high-frequency content (sharper edges, finer texture, or compression block artifacts)."""
    f = np.fft.fft2(gray.astype(np.float64))
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    H, W = gray.shape
    cy, cx = H // 2, W // 2
    ry, rx = max(1, int(H * low_frac / 2)), max(1, int(W * low_frac / 2))
    low_energy = mag[cy - ry:cy + ry, cx - rx:cx + rx].sum()
    total_energy = mag.sum() + 1e-8
    return float(1.0 - (low_energy / total_energy))


def compute_image_stats(img_path: Path) -> Optional[Dict]:
    try:
        pil_img = Image.open(img_path)
        width, height = pil_img.size
        file_size_kb = img_path.stat().st_size / 1024.0
        jpeg_qmean = estimate_jpeg_quant_mean(pil_img)

        gray = np.array(pil_img.convert('L'))
        laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        fft_ratio = compute_fft_high_freq_ratio(gray)
        noise_sigma = compute_noise_sigma(gray)
        mean_intensity = float(gray.mean())
        std_intensity = float(gray.std())

        return dict(
            width=width, height=height, aspect_ratio=width / height,
            file_size_kb=file_size_kb, jpeg_qmean=jpeg_qmean,
            laplacian_var=laplacian_var, fft_high_freq_ratio=fft_ratio,
            noise_sigma=noise_sigma, mean_intensity=mean_intensity,
            std_intensity=std_intensity,
        )
    except Exception as e:
        print(f'  Warning: failed on {img_path}: {e}')
        return None


STAT_COLUMNS = [
    'width', 'height', 'aspect_ratio', 'file_size_kb', 'jpeg_qmean',
    'laplacian_var', 'fft_high_freq_ratio', 'noise_sigma',
    'mean_intensity', 'std_intensity',
]
print(f'{len(STAT_COLUMNS)} statistics defined: {STAT_COLUMNS}')

## 5. Compute Statistics

In [ ]:
def compute_stats_for_df(df: pd.DataFrame, label_name: str) -> pd.DataFrame:
    records = []
    t0 = time.time()
    for i, (_, row) in enumerate(df.iterrows()):
        img_path = resolve_image_path(data_dir, row['image_path'])
        stats = compute_image_stats(img_path)
        if stats is not None:
            stats.update(id=row['id'], type=row['type'], label=row['label'])
            records.append(stats)
        if (i + 1) % 250 == 0:
            print(f'  [{label_name}] {i+1}/{len(df)} processed, {time.time()-t0:.0f}s elapsed')
    elapsed = time.time() - t0
    result = pd.DataFrame(records)
    print(f'[{label_name}] done: {len(result)}/{len(df)} images in {elapsed:.0f}s')
    return result


print('Computing stats on GENUINE sample...')
stats_genuine = compute_stats_for_df(sample_genuine, 'genuine')

print('\nComputing stats on FRAUD sample (secondary pass)...')
stats_fraud = compute_stats_for_df(sample_fraud, 'fraud')

stats_genuine.to_csv(f'{OUTPUT_DIR}/stats_genuine.csv', index=False)
stats_fraud.to_csv(f'{OUTPUT_DIR}/stats_fraud.csv', index=False)
print(f'\nSaved raw stats to {OUTPUT_DIR}/')

## 6. Cross-Type Comparison — Genuine Images

For each statistic: per-type summary (mean/median/std), a Kruskal-Wallis test across all 5 type groups (non-parametric — doesn't assume normality, appropriate here), and a focused Mann-Whitney U test contrasting `MAURITIUS/ID` (the ID-card format) against all pooled DL-format types — mirroring the exact split seen in the Grad-CAM buckets.

In [ ]:
def cross_type_report(stats_df: pd.DataFrame, title: str) -> pd.DataFrame:
    print(f'\n{"=" * 70}')
    print(f'{title}')
    print(f'{"=" * 70}')

    results = []
    id_types = ['MAURITIUS/ID']
    dl_types = [t for t in stats_df['type'].unique() if t not in id_types]

    for col in STAT_COLUMNS:
        groups = [stats_df[stats_df['type'] == t][col].dropna().values
                  for t in stats_df['type'].unique()]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            continue
        try:
            h_stat, p_kw = scipy.stats.kruskal(*groups)
        except Exception:
            h_stat, p_kw = np.nan, np.nan

        id_vals = stats_df[stats_df['type'].isin(id_types)][col].dropna().values
        dl_vals = stats_df[stats_df['type'].isin(dl_types)][col].dropna().values
        if len(id_vals) > 0 and len(dl_vals) > 0:
            try:
                u_stat, p_mw = scipy.stats.mannwhitneyu(id_vals, dl_vals, alternative='two-sided')
            except Exception:
                u_stat, p_mw = np.nan, np.nan
        else:
            u_stat, p_mw = np.nan, np.nan

        results.append({
            'statistic': col,
            'kruskal_p (all 5 types)': p_kw,
            'mannwhitney_p (ID vs pooled DL)': p_mw,
            'ID_mean': id_vals.mean() if len(id_vals) else np.nan,
            'DL_mean': dl_vals.mean() if len(dl_vals) else np.nan,
        })

    report = pd.DataFrame(results).sort_values('mannwhitney_p (ID vs pooled DL)')
    pd.set_option('display.float_format', lambda x: f'{x:.4g}')
    print(report.to_string(index=False))

    sig = report[report['mannwhitney_p (ID vs pooled DL)'] < 0.001]
    print(f'\n{len(sig)}/{len(report)} statistics differ significantly (p<0.001) between '
          f'MAURITIUS/ID and the pooled DL types.')
    if len(sig) > 0:
        print('Significant statistics:', sig['statistic'].tolist())
    return report


report_genuine = cross_type_report(stats_genuine, 'GENUINE images (label=0) — cross-type comparison')

## 7. Cross-Type Comparison — Fraud Images (secondary pass)

Same test, run on `label=1` images. If the same statistics come out significant here too, the signature is present **regardless of label** — consistent with a rendering/capture pipeline artifact rather than anything to do with how fraud was synthesized.

In [ ]:
report_fraud = cross_type_report(stats_fraud, 'FRAUD images (label=1) — cross-type comparison')

## 8. Visualization — Boxplots by Type

In [ ]:
def plot_stats_by_type(stats_df: pd.DataFrame, title: str, filename: str) -> None:
    n_stats = len(STAT_COLUMNS)
    ncols = 2
    nrows = (n_stats + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows))
    axes = axes.flatten()

    types_sorted = sorted(stats_df['type'].unique())
    for ax, col in zip(axes, STAT_COLUMNS):
        data = [stats_df[stats_df['type'] == t][col].dropna().values for t in types_sorted]
        bp = ax.boxplot(data, labels=types_sorted, showfliers=False, patch_artist=True)
        for patch, t in zip(bp['boxes'], types_sorted):
            patch.set_facecolor('#f4a259' if t == 'MAURITIUS/ID' else '#3b6ba5')
            patch.set_alpha(0.7)
        ax.set_title(col, fontsize=10)
        ax.tick_params(axis='x', rotation=45, labelsize=7)

    for ax in axes[len(STAT_COLUMNS):]:
        ax.axis('off')

    fig.suptitle(f'{title}  (orange = MAURITIUS/ID, blue = DL-format types)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {OUTPUT_DIR}/{filename}')


plot_stats_by_type(stats_genuine, 'GENUINE images — stats by document type', 'boxplots_genuine.png')
plot_stats_by_type(stats_fraud,   'FRAUD images — stats by document type',   'boxplots_fraud.png')

## 9. Summary and Reading Guide

In [ ]:

print('=' * 70)
print('SUMMARY')
print('=' * 70)

sig_genuine = set(report_genuine[report_genuine['mannwhitney_p (ID vs pooled DL)'] < 0.001]['statistic'])
sig_fraud   = set(report_fraud[report_fraud['mannwhitney_p (ID vs pooled DL)'] < 0.001]['statistic'])
sig_both    = sig_genuine & sig_fraud

print(f'Significant (p<0.001) ID-vs-DL differences, GENUINE images: {sorted(sig_genuine)}')
print(f'Significant (p<0.001) ID-vs-DL differences, FRAUD images  : {sorted(sig_fraud)}')
print(f'Significant in BOTH (label-independent)                   : {sorted(sig_both)}')

print()
print('-' * 70)
print('READING GUIDE')
print('-' * 70)
print('- If width/height/aspect_ratio differ sharply: the ID-card and DL templates were likely')
print('  rendered/scanned at different base resolutions -- an easy, obvious pipeline tell that')
print('  has nothing to do with fraud. Fix: resize/resolution-jitter augmentation to scrub it.')
print()
print('- If jpeg_qmean differs sharply: the two format pipelines used different JPEG compression')
print('  settings somewhere upstream. Fix: widen AUG_P_JPEG quality range substantially, and make')
print('  sure it is applied with high probability regardless of format.')
print()
print('- If noise_sigma or fft_high_freq_ratio differ sharply: one pipeline is systematically')
print('  noisier/sharper than the other (e.g. one used a different renderer, scan process, or')
print('  recapture-simulation step upstream of your own augmentation). Fix: stronger/broader')
print('  noise and blur augmentation ranges.')
print()
print('- If the signature is significant in BOTH genuine and fraud samples: strong evidence this')
print('  is a pipeline/rendering artifact, not a fraud-specific cue -- the model can shortcut on')
print('  "which pipeline produced this image" without ever looking at forgery evidence.')
print()
print('- If nothing comes out significant: the pixel-statistics explanation is NOT supported, and')
print('  the format-correlated Grad-CAM pattern likely reflects something more semantic (e.g. the')
print('  model has genuinely learned different, format-specific decision rules that do not')
print('  transfer well) rather than a low-level fingerprint -- worth revisiting with per-type')
print('  Grad-CAM comparisons on TRAINING images next, rather than augmentation changes.')
